# 🎬 AutoVideo - Kaggle Notebook Setup (Fixed Cloudflare DNS)

Auto Recap & Video Translation API on Kaggle GPU Instance with Public Web Tunnel.

### ⚠️ Required Settings:
1. **Accelerator**: GPU T4 or P100
2. **Internet**: ON

In [ ]:
# 1. Git Clone Project from GitHub & Install Dependencies
!nvidia-smi
!apt-get update -qq && apt-get install -y -qq ffmpeg

import os
REPO_URL = "https://github.com/surviveman78-commits/autovideo.git"

if not os.path.exists('/kaggle/working/autovideo'):
    !git clone {REPO_URL} /kaggle/working/autovideo

%cd /kaggle/working/autovideo
!pip install -q -r requirements.txt

In [ ]:
# 2. Setup F5-Myanmar TTS v2 Engine for GPU Voice Clone
import os
!pip install -q f5-tts

print("Current Directory:", os.getcwd())
print("Files:", os.listdir('.'))

In [ ]:
# 3. Set API Keys (Optional - Can also be set directly in Web UI)
import os
os.environ["GROQ_API_KEY"] = "YOUR_GROQ_API_KEY_HERE"
os.environ["GEMINI_API_KEY"] = "YOUR_GEMINI_API_KEY_HERE"
print("API Keys set in environment!")

In [ ]:
# 3.5 Upload Local Video File or Copy from Kaggle Dataset
import os
import shutil
from IPython.display import HTML, display

# Option A: If you uploaded a video via Kaggle 'Add Data' (/kaggle/input/...), set path below:
KAGGLE_INPUT_VIDEO = ""  # Example: "/kaggle/input/my-video-dataset/sample.mp4"

if KAGGLE_INPUT_VIDEO and os.path.exists(KAGGLE_INPUT_VIDEO):
    os.makedirs("downloads/video", exist_ok=True)
    target_path = os.path.join("downloads/video", os.path.basename(KAGGLE_INPUT_VIDEO))
    shutil.copy(KAGGLE_INPUT_VIDEO, target_path)
    print(f"✅ Video copied successfully to: {target_path}")

# Option B: Interactive Browser File Uploader Widget
uploader_html = '''
<div style="border: 2px dashed #6366f1; padding: 20px; text-align: center; border-radius: 12px; background: rgba(99, 102, 241, 0.05); font-family: sans-serif;">
    <h3 style="margin-top:0; color: #4f46e5;">🎬 Upload Local Video File directly to AutoVideo</h3>
    <p style="color: #6b7280; font-size: 0.9rem;">Select an MP4, MOV, or MKV video file from your computer:</p>
    <input type="file" id="localVideoFile" accept="video/*" style="margin: 10px 0; padding: 6px; border: 1px solid #ccc; border-radius: 6px;">
    <br>
    <button onclick="uploadVideoToAutoVideo()" style="background: #4f46e5; color: white; border: none; padding: 10px 24px; border-radius: 8px; cursor: pointer; font-weight: bold; margin-top: 10px;">🚀 Upload Video</button>
    <p id="uploadStatusMsg" style="margin-top: 12px; font-weight: bold;"></p>
</div>

<script>
async function uploadVideoToAutoVideo() {
    const input = document.getElementById('localVideoFile');
    const status = document.getElementById('uploadStatusMsg');
    if (!input.files.length) {
        status.innerText = '❌ Please select a video file first!';
        status.style.color = '#ef4444';
        return;
    }
    const file = input.files[0];
    const formData = new FormData();
    formData.append('file', file);
    status.innerText = '⏳ Uploading ' + file.name + ' (' + (file.size / (1024*1024)).toFixed(1) + ' MB)... Please wait!';
    status.style.color = '#4f46e5';
    try {
        const res = await fetch('/api/upload', {
            method: 'POST',
            body: formData
        });
        const data = await res.json();
        if (data.status === 'success') {
            status.innerText = '✅ Uploaded successfully! Saved at: ' + data.video_path;
            status.style.color = '#10b981';
        } else {
            status.innerText = '❌ Upload failed!';
            status.style.color = '#ef4444';
        }
    } catch(e) {
        status.innerText = '⚠️ Note: You can also use the Cloudflare Public Web UI link (Cell 4) to upload videos!';
        status.style.color = '#f59e0b';
    }
}
</script>
'''
display(HTML(uploader_html))


In [ ]:
# 4. Launch FastAPI Web App & Cloudflare Tunnel with DNS Wait
import os
import re
import time
import subprocess
import sys

# Clean up any previous background processes to free up port 8000
subprocess.run("pkill -9 -f uvicorn || true", shell=True)
subprocess.run("pkill -9 -f cloudflared || true", shell=True)
time.sleep(1)

if not os.path.exists("./cloudflared"):
    print("📥 Downloading Cloudflare Tunnel Binary...")
    subprocess.run(["curl", "-L", "--output", "cloudflared", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"], check=True)
    subprocess.run(["chmod", "+x", "cloudflared"], check=True)

print("🚀 Starting AutoVideo FastAPI Server...")
with open("uvicorn.log", "w") as uvicorn_log:
    server_process = subprocess.Popen(
        [sys.executable, "-m", "uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"],
        stdout=uvicorn_log,
        stderr=uvicorn_log
    )
time.sleep(4)

print("🌐 Opening Public Web Tunnel...")
with open("cloudflared.log", "w") as cf_log:
    tunnel_process = subprocess.Popen(
        ["./cloudflared", "tunnel", "--url", "http://localhost:8000"],
        stdout=cf_log,
        stderr=cf_log
    )

print("⏳ Establishing tunnel & registering DNS (waiting 6s)...")
time.sleep(6)

public_url = None
if os.path.exists("cloudflared.log"):
    with open("cloudflared.log", "r") as f:
        content = f.read()
        urls = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", content)
        if urls:
            public_url = urls[-1]

if public_url:
    print("\n" + "="*70)
    print("🎉 AUTOVIDEO WEB UI IS LIVE AT:")
    print(f"👉 {public_url}")
    print("="*70)
    print("⚠️ NOTE: If you see 'DNS_PROBE_FINISHED_NXDOMAIN', wait 10-15s and refresh!")
    print("="*70 + "\n")
else:
    print("⚠️ Could not extract URL automatically. Checking logs below:")
    if os.path.exists("uvicorn.log"):
        print("--- uvicorn.log ---")
        with open("uvicorn.log", "r") as f:
            print(f.read())
    if os.path.exists("cloudflared.log"):
        print("--- cloudflared.log ---")
        with open("cloudflared.log", "r") as f:
            print(f.read())

In [ ]:
# 5. Check Rendered Videos & Download
import os
from IPython.display import FileLink, display

recap_dir = "downloads/recap"
if os.path.exists(recap_dir):
    for f in os.listdir(recap_dir):
        file_path = os.path.join(recap_dir, f)
        print(f"- {f} ({os.path.getsize(file_path)/(1024*1024):.2f} MB)")
        display(FileLink(file_path))
else:
    print("No rendered videos found yet.")